In [1]:
%load_ext autoreload
%autoreload 2

In [17]:
import sys
sys.path.append('../../MARS')
from MARS.run_lars.utils import EarlyStopping, LRScheduler
import os
import pandas as pd
import torch.backends.cudnn as cudnn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import numpy as np
import time
import tqdm
import shutil
import yaml
import sys
from datetime import datetime 
import time
import random
import wandb
from matplotlib import pyplot as plt



from MARS.run_lars import dataset
from MARS.run_lars import utils
from MARS.run_lars import logging_utils


In [24]:
model = utils.model_prediction('/gpfs/data/mayerhoeferlab/projects/lars/lars-avg-max_chpt/LARS_trained_state-dict_0.pth', 'resnet34')
model = model.cuda()
model.eval()


my_model(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tru

In [31]:
def expand_resnet_model_to_two_channels(model_a):
    newconv = nn.Conv2d(2, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    newweights = torch.zeros((64,2,7,7))
    newweights[:,0,:,:] = model_a.features[0].weight[:,0,:,:]
    newweights[:,1,:,:] = model_a.features[0].weight[:,0,:,:]
    newconv.weight = nn.Parameter(newweights)
    model_a.features[0] = newconv
    return model_a

In [25]:
data_path_img='/gpfs/data/mayerhoeferlab/projects/myeloma/process_dicoms_for_mars/images_h5/'
data_path='/gpfs/data/mayerhoeferlab/projects/myeloma/datafiles_for_mars/train_df.csv'
traindf =dataset.get_datasets('train', norm=True,transform=None, data_path_img=data_path_img, data_path=data_path,
                postfilter_fwhm_xy_mm=None, normalize_mean=None, normalize_std=None, target_label_name = 'PET_jco',
                postfilter_fwhm_xy_mm_min=None, postfilter_fwhm_xy_mm_max=None,
                suv_clip_threshold_min=None, suv_clip_threshold_max=None, combine_mipviews='feature_concatenation')

In [ ]:
model2 = utils.modify_resnet_to_accomodate_two_mipviews(model, mipview_strategy='feature_concatenation', feature_concat_divide_weights = True)
model2 = model2.cuda()
model2.eval()

my_model_feat_fusion(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runni